In [1]:
import json
import multiprocessing
import openai
import os
import os.path as osp
import shutil
import sys
import time
import torch
from aider.coders import Coder
from aider.io import InputOutput
from aider.models import Model
from datetime import datetime
from generate_ideas import generate_ideas

In [3]:
def print_time():
    print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
NUM_REFLECTIONS = 3


OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
client_model = 'gpt-4o-mini'
client = openai.OpenAI(api_key=OPENAI_API_KEY)

base_dir = osp.abspath('./templates')
results_dir = osp.abspath('./results')

skip_generation = True

In [5]:
ideas = generate_ideas(
        base_dir,
        client=client,
        model=client_model,
        skip_generation=skip_generation,
        max_num_generations=2,
        num_reflections=NUM_REFLECTIONS,
    )

Skipping idea generation


In [9]:
with open(osp.join(base_dir, 'prompt.json'), 'r') as f:
    prompt = json.load(f)
    
idea_system_prompt = prompt['system']

In [10]:
idea_system_prompt

'You are an ambitious AI PhD student who is looking to publish a paper that will contribute significantly to the field.'

In [11]:
with open(osp.join(base_dir, 'seed_ideas.json'), 'r') as f:
    seed_ideas = json.load(f)

In [13]:
idea_str_archive = []
for seed_idea in seed_ideas:
    idea_str_archive.append(json.dumps(seed_idea))

In [14]:
with open(osp.join(base_dir, 'experiment.py'), 'r') as f:
    code = f.read()

In [17]:
prev_ideas_string = '\n\n'.join(idea_str_archive)

In [21]:
from prompts.prompts import idea_first_prompt, idea_reflection_prompt

In [19]:
msg_history = []

In [23]:
num_reflections = 2

msg = idea_first_prompt.format(
                    task_description=prompt["task_description"],
                    code=code,
                    prev_ideas_string=prev_ideas_string,
                    num_reflections=num_reflections,
                )

In [29]:
msg_history

[]

In [28]:
idea_system_prompt

'You are an ambitious AI PhD student who is looking to publish a paper that will contribute significantly to the field.'

In [27]:
client_model

'gpt-4o-mini'

In [37]:
from llm import get_response_from_llm

In [38]:
(text,
 msg_history) = get_response_from_llm(
    msg,
    client=client,
    model=client_model,
    system_message=idea_system_prompt,
    msg_history=msg_history,
)

In [40]:
print(text)

THOUGHT:
In the current implementation, the trading environment employs a simple reward structure based primarily on profit and losses from trades. However, the market is inherently complex and volatile, and additional factors such as risk management and drawdown control can play a crucial role in trading performance. I propose to enhance the existing framework by incorporating a risk-adjusted reward system. This system can utilize metrics like the Sharpe Ratio or Sortino Ratio to evaluate performance while penalizing excessive risk. 

The high-level plan involves modifying the reward calculation within the `step` function of the `TradingEnv` class to include a risk-reward balance. This will involve keeping track of historical returns and volatility to compute the risk-adjusted reward. The outcomes will be measured not only in terms of total profit but also in terms of the Sharpe Ratio over episodes, giving a more holistic view of the agent's performance.

This idea is different from t

In [6]:
novel_ideas = [idea for idea in ideas]

In [7]:
idea = novel_ideas[0]

In [8]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
idea_name_ = idea['Name']
idea_name = f'{timestamp}_{idea_name_}'
folder_name = osp.join(results_dir, idea_name)
idea_name, folder_name

('20250228_191413_gru_based_rl_actor_critic',
 '/home/yvasiliuk/trader/ai/results/20250228_191413_gru_based_rl_actor_critic')

In [ ]:
destination_dir = folder_name
shutil.copytree(base_dir, destination_dir, dirs_exist_ok=True)

In [ ]:
with open(osp.join(base_dir, 'run_0', 'final_info.json'), 'r') as f:
    baseline_results = json.load(f)

In [ ]:
baseline_results = {k: v['means'] for k, v in baseline_results.items()}
baseline_results

In [ ]:
exp_file = osp.join(folder_name, 'experiment.py')
vis_file = osp.join(folder_name, 'plot.py')
notes = osp.join(folder_name, 'notes.txt')

In [ ]:
with open(notes, "w") as f:
    f.write(f"# Title: {idea['Title']}\n")
    f.write(f"# Experiment description: {idea['Experiment']}\n")
    f.write(f"## Run 0: Baseline\n")
    # f.write(f"Results: {baseline_results}\n")
    f.write(f"Description: Baseline results.\n")


In [ ]:
print_time()
print(f"*Starting idea: {idea_name}*")
## PERFORM EXPERIMENTS
fnames = [exp_file, vis_file, notes]
io = InputOutput(
    yes=True, chat_history_file=f"{folder_name}/{idea_name}_aider.txt"
)

In [ ]:
main_model = Model(model)

coder = Coder.create(
        main_model=main_model,
        fnames=fnames,
        io=io,
        stream=False,
        use_git=False,
        edit_format="diff",
    )

In [1]:
import os

In [2]:
os.path.expanduser('~/trader/data/BTC-2021min.csv')

'/home/yvasiliuk/trader/data/BTC-2021min.csv'